In [ ]:
!pip install deepface

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
###################
##  Code cell 1  ##
###################

from IPython.display import display, Javascript
from google.colab.output import eval_js

# Function to use a webcam.
#   [Parameters]
#     quality : The quality of image used to generate a jpeg image
#   [Returns]
#     None
#   Reference: https://qiita.com/a2kiti/items/f32de4f51a31d609e5a5
def use_cam(quality=0.8):
  js = Javascript('''
    async function useCam(quality) {
      const div = document.createElement('div');
      document.body.appendChild(div);
      //video element
      const video = document.createElement('video');
      video.style.display = 'None';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      //canvas for display. frame rate is depending on display size and jpeg quality.
      const canvas = document.createElement('canvas');
      canvas.width  = video.videoWidth;
      canvas.height = video.videoHeight;
      const canvasCtx = canvas.getContext('2d');
      div.appendChild(canvas);

      // A text area (paragraph) used to show the current emotion
      const text = document.createElement('p');
      document.body.appendChild(text);
      text.innerHTML = 'Your current emotion is <span style="color:red">unknown</span>.';

      //exit button
      const btn_div = document.createElement('div');
      document.body.appendChild(btn_div);
      const exit_btn = document.createElement('button');
      exit_btn.textContent = 'Exit';
      var exit_flg = true;
      exit_btn.onclick = function() {exit_flg = false};
      btn_div.appendChild(exit_btn);

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      var send_num = 0;
      // loop
      _canvasUpdate();
      async function _canvasUpdate() {
            canvasCtx.drawImage(video, 0, 0);
            if (send_num<1){
                send_num += 1;
                const img = canvas.toDataURL('image/jpeg', quality);
                const result = google.colab.kernel.invokeFunction('notebook.run', [img], {}); // Invoke the Python function: run(img)
                result.then(function(value) { // value: a JSON object returned by the Python function "run"
                    send_num -= 1;
                    const emotion = value.data['application/json'].emotion;
                    text.innerHTML = 'Your emotion is <span style="color:red">' + emotion + '</span>.';
                });
            }
            if (exit_flg){
                requestAnimationFrame(_canvasUpdate);
            }else{
                stream.getVideoTracks()[0].stop();
            }
      };
    }
    ''')
  display(js)
  data = eval_js('useCam({})'.format(quality))

In [ ]:
###################
##  Code cell 2  ##
###################

from base64 import b64decode
from deepface import DeepFace
from deepface.modules.exceptions import FaceNotDetected
from google.colab import output
import numpy as np
import IPython

# A temporary file used for emotion estimation
TEMP_FILE_NAME = 'tmp.jpg'

# Function to run the program
#   [Parameters]
#     img_str : An image data (camera capture) represented as a Base64 string
#   [Returns]
#     A JSON object including the information of the emotion
def run(img_str):
  # Extract the image data from Base64 string and convert it into an image (jpeg) data.
  binary = b64decode(img_str.split(',')[1])
  # Save the image data into the temporary file.
  with open(TEMP_FILE_NAME, 'wb') as f:
    f.write(binary)
  # Estimate the emotion using the temporary file.
  try:
    emotion = DeepFace.analyze(img_path = TEMP_FILE_NAME, actions = ['emotion'])[0]['dominant_emotion']
  except FaceNotDetected:
    emotion = 'unknown'
  # Return a JSON object including the information of the estimated emotion.
  return IPython.display.JSON({'emotion' : emotion})

# Register a callback for the real-time estimation.
output.register_callback('notebook.run', run)

In [ ]:
###################
##  Code cell 3  ##
###################

# Start the program.
use_cam()